In [1]:
%load_ext autoreload
%autoreload 2

# RNN Model Training

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import joblib
import matplotlib.dates as mdates
from sklearn.metrics import  r2_score
# set up relative imports
project_folder = Path.cwd().parent.parent
sys.path.append(str(project_folder))

In [3]:
from modeling.model.lstm import create_lstm_model, get_callbacks
from modeling.utilities.data_prep import setup_sequential_data

#### read in data

In [ ]:
data_folder = Path(r'..\..\data\model-ready\water-weather')

# train
train_X_df = pd.read_parquet(data_folder / 'scaled-X-train.parquet')
train_y_df = pd.read_parquet(data_folder / 'scaled-Y-train.parquet') # the Y data is not actually scaled inspite of the name

# test
test_X_df = pd.read_parquet(data_folder / 'scaled-X-test.parquet')
test_y_df = pd.read_parquet(data_folder / 'scaled-Y-test.parquet') # the Y data is not actually scaled inspite of the name

# val
val_X_df = pd.read_parquet(data_folder / 'scaled-X-val.parquet')
val_y_df = pd.read_parquet(data_folder / 'scaled-Y-val.parquet') # the Y data is not actually scaled inspite of the name



In [5]:
scaler = joblib.load(data_folder / 'minmax_scaler.joblib')

#### set up sequential data

In [6]:
# train
train_X_df.reset_index(drop=True, inplace=True)
train_y_df.reset_index(drop=True, inplace=True)

# test
test_X_df.reset_index(drop=True, inplace=True)
test_y_df.reset_index(drop=True, inplace=True)

# val
val_X_df.reset_index(drop=True, inplace=True)
val_y_df.reset_index(drop=True, inplace=True)

In [7]:
seq_size = 10
X_train, y_train, y_idx_train = setup_sequential_data(train_X_df, train_y_df, seq_size)
X_val, y_val, y_idx_val =  setup_sequential_data(val_X_df, val_y_df, seq_size)
X_test, y_test, y_idx_test =  setup_sequential_data(test_X_df, test_y_df, seq_size)

In [8]:
X_test.shape

(4731, 10, 16)

### Compile and Train the model

In [9]:
input_shape = (X_train.shape[1], X_train.shape[2])

In [10]:
model = create_lstm_model(input_shape)

In [11]:
model.compile(optimizer='adam', loss='mse',)

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 10, 50)         │        13,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,651 (131.45 KB)

 Trainable params: 33,651 (131.45 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X_train, 
          y_train, 
          epochs=100, 
          batch_size=32,
          validation_data=(X_val, y_val),
          callbacks=get_callbacks()
          )

Epoch 1/100
691/691 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 21.7541 - val_loss: 7.5116 - learning_rate: 0.0010
Epoch 2/100
691/691 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 8.4571 - val_loss: 7.6029 - learning_rate: 0.0010
Epoch 3/100
691/691 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 3.9261 - val_loss: 1.0361 - learning_rate: 0.0010
Epoch 4/100
691/691 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - loss: 2.4143 - val_loss: 0.9700 - learning_rate: 0.0010
Epoch 5/100
691/691 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - loss: 2.1396 - val_loss: 0.7546 - learning_rate: 0.0010
Epoch 6/100
691/691 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - loss: 2.0072 - val_loss: 0.4844 - learning_rate: 0.0010
Epoch 7/100
691/691 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - loss: 1.9343 - val_loss: 0.4596 - learning_rate: 0.0010
Epoch 8/100
691/691 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 1.8987 - val_loss: 0.4580 - learning_rate: 0.0010
Epoch 9/100
691/691 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 1.8908 - val_loss: 0.4165 - learning_r

#### Evaluate the model

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
test_x_unscaled = scaler.inverse_transform(test_X_df)
test_x_df = pd.DataFrame(columns=test_X_df.columns, data=test_x_unscaled)

In [ ]:
test_x_df.shape

In [ ]:
results_df = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred.flatten()
}, index=y_idx_test)

In [ ]:
results_df

In [ ]:
test_x_df = test_x_df.join(results_df,
                how='inner')

In [ ]:
# this date recreation is imperfect because of how we encoded day
test_x_df['date'] = pd.to_datetime(test_x_df[['year','month','day']], errors='coerce')

In [ ]:
test_x_df['errors'] = test_x_df.predicted - test_x_df.actual

In [ ]:
plt.hist(test_x_df.errors, bins='auto')
plt.title('LSTM')
plt.xlabel('Error (\u00b0C)', fontsize=16)
plt.show()

In [ ]:
pd.DataFrame(test_x_df.errors.abs().describe()).T.round(3)

In [ ]:
plot_df = test_x_df[:30].copy()
fig, ax = plt.subplots(figsize=(10,5))

ax.plot(plot_df.date, plot_df.actual, label='True', color='cornflowerblue')
ax.plot(plot_df.date, plot_df.predicted, label='Predicted', color='firebrick')
ax.set_ylabel('Temperature (\u00b0C)', fontsize=16)
ax.set_xlabel('Date', fontsize=16)
ax.set_title('LSTM | Example Prediction', fontsize=18)
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax.tick_params(axis='x', labelrotation=45)
fig.text(.9, -.05, f'{int(plot_df.year.unique()[0])}')
plt.show()

In [ ]:
mse = np.mean([x**2 for x in test_x_df.errors])
rmse = np.sqrt(mse)
r_2 = r2_score(y_test, y_pred)

print(f'RMSE = {round(rmse, 2)}')
print(f'r2 score = {round(r_2, 2)}')

##### RMSE and R2 Score are 0.72 and 0.93 respectively for the LSTM model trained on the no wind dataset. This is actually worse than our baseline linear regression model